# DeepTopic motif co-occurrence and peak composition

Analyze which motifs co-occur in the same peaks — reveals combinatorial TF logic
driving each topic's regulatory program.

Uses seqlet-level hit tables from notebook 6 (MoDISco-based, attribution-aware).
Adapted from `bin/3_single_task_models/7_motif_cooccurrence.ipynb` (ChromBPNet).

**Analyses:**
1. Per-topic peak x motif binary matrix -> motif co-occurrence (Jaccard)
2. Average co-occurrence across topics
3. Topic-specific vs shared co-occurrence patterns
4. Peak complexity (how many motifs per peak)

**Inputs:**
- `crested_model/motifs/hits/per_topic/Topic*.hits.tsv` — seqlet-level hit tables

**Outputs:**
- `crested_model/motifs/cooccurrence/avg_cooccurrence_matrix.csv`
- `crested_model/motifs/cooccurrence/avg_cooccurrence_clustermap.pdf`
- `crested_model/motifs/cooccurrence/peak_complexity_stacked.pdf`
- `crested_model/motifs/cooccurrence/cooccurrence_grid.pdf`
- `crested_model/motifs/cooccurrence/motif_pair_cooccurrence.csv`

# Set-up

In [ ]:
import os
import numpy as np
import pandas as pd
from collections import Counter

import matplotlib
import seaborn as sns
import matplotlib.pyplot as plt
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [ ]:
# Paths
dataset = 'sc-islet-differentiation_10X-Multiome'
subset = 'endocrine_replicate'
base_dir = f'/cellar/users/aklie/data/datasets/{dataset}'
results_dir = f'{base_dir}/results/4_topic_models/{subset}'
motif_dir = f'{results_dir}/crested_model/motifs'
hits_dir = f'{motif_dir}/hits/per_topic'

path_out = f'{motif_dir}/cooccurrence'
os.makedirs(path_out, exist_ok=True)

# Annotations
initial_motifs = pd.read_csv(f'{motif_dir}/tfs_initial.txt', sep='\t', header=None, skiprows=0,
                              names=['cluster_name', 'annotation', 'evalue'])
annot_short = {k: v.split('_')[0] for k, v in
               initial_motifs.set_index('cluster_name')['annotation'].to_dict().items()}

# Topic categories
topic_cats = pd.read_csv(f'{results_dir}/summary/topic_categories.tsv', sep='\t')
cat_colors = {
    'one-to-one': '#2ca02c', 'lineage': '#1f77b4', 'shared': '#ff7f0e',
    'structural': '#7f7f7f', 'batch-driven': '#d62728', 'unannotated': '#bcbd22',
}
topic_cat_dict = dict(zip(topic_cats['topic'], topic_cats['category']))

all_topics = [f'Topic{i}' for i in range(1, 51)]

# Build per-topic peak x motif binary matrices

In [ ]:
peak_motif_binary = {}  # topic -> binary DataFrame (peaks x motifs)
peak_complexity = {}    # topic -> Series of motifs-per-peak
topic_list = []

for topic in all_topics:
    hits_path = f'{hits_dir}/{topic}.hits.tsv'
    if not os.path.exists(hits_path):
        continue

    df = pd.read_csv(hits_path, sep='\t', usecols=['peak_idx', 'cluster'])
    if len(df) == 0:
        continue

    binary = df.groupby(['peak_idx', 'cluster']).size().unstack(fill_value=0)
    binary = (binary > 0).astype(int)

    peak_motif_binary[topic] = binary
    peak_complexity[topic] = binary.sum(axis=1)
    topic_list.append(topic)

    print(f'{topic}: {binary.shape[0]} peaks, {binary.shape[1]} motifs, '
          f'median {peak_complexity[topic].median():.0f} motifs/peak')

# Union of all motif names
all_motifs = sorted(set().union(*[set(b.columns) for b in peak_motif_binary.values()]))
print(f'\n{len(all_motifs)} unique motifs across {len(topic_list)} topics')

# Peak complexity: how many motifs per peak?

In [ ]:
max_motifs = 8
prop_data = {}
for topic in topic_list:
    counts = peak_complexity[topic].clip(upper=max_motifs).value_counts(normalize=True).sort_index()
    prop_data[topic] = counts

prop_df = pd.DataFrame(prop_data).T.fillna(0)
prop_df = prop_df.reindex(topic_list)
prop_df.columns = [f'{int(c)}' if c < max_motifs else f'{max_motifs}+' for c in prop_df.columns]

fig, ax = plt.subplots(figsize=(16, 5))
prop_df.plot.bar(stacked=True, ax=ax, colormap='YlOrRd', width=0.85, edgecolor='white', linewidth=0.3)
ax.set_ylabel('Fraction of peaks')
ax.set_title('Peak complexity: motifs per peak')
ax.legend(title='# motifs', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.savefig(f'{path_out}/peak_complexity_stacked.pdf', dpi=300, bbox_inches='tight')
plt.show()

print('Mean motifs per peak:')
for topic in topic_list[:10]:
    print(f'  {topic:12s} {peak_complexity[topic].mean():.2f}')

# Motif co-occurrence matrices (Jaccard)

In [ ]:
cooccur_matrices = {}

for topic in topic_list:
    binary = peak_motif_binary[topic].reindex(columns=all_motifs, fill_value=0)

    # Filter to motifs with > 10 peaks
    motif_counts = binary.sum(axis=0)
    keep = motif_counts[motif_counts > 10].index.tolist()
    binary_filt = binary[keep]

    n = len(keep)
    jaccard = pd.DataFrame(np.zeros((n, n)), index=keep, columns=keep)

    vals = binary_filt.values
    for i in range(n):
        for j in range(i, n):
            a, b = vals[:, i], vals[:, j]
            intersection = (a & b).sum()
            union = (a | b).sum()
            jac = intersection / union if union > 0 else 0
            jaccard.iloc[i, j] = jac
            jaccard.iloc[j, i] = jac

    cooccur_matrices[topic] = jaccard

print(f'Computed co-occurrence for {len(cooccur_matrices)} topics')

# Average co-occurrence across topics

In [ ]:
# Motifs present in >= 50% of topics
motif_presence = Counter()
for mat in cooccur_matrices.values():
    for m in mat.index:
        motif_presence[m] += 1

min_topics = len(cooccur_matrices) * 0.5
common_motifs = sorted([m for m, c in motif_presence.items() if c >= min_topics])
print(f'Motifs in >= 50% of topics (>10 peaks each): {len(common_motifs)}')

avg_cooccur = pd.DataFrame(np.zeros((len(common_motifs), len(common_motifs))),
                            index=common_motifs, columns=common_motifs)
ct_count = pd.DataFrame(np.zeros_like(avg_cooccur.values),
                         index=common_motifs, columns=common_motifs)

for topic, mat in cooccur_matrices.items():
    shared = [m for m in common_motifs if m in mat.index]
    avg_cooccur.loc[shared, shared] += mat.loc[shared, shared]
    ct_count.loc[shared, shared] += 1

avg_cooccur = avg_cooccur / ct_count.clip(lower=1)
labels = [annot_short.get(m, m) for m in common_motifs]

g = sns.clustermap(
    avg_cooccur.values,
    cmap='YlOrRd',
    figsize=(14, 14),
    xticklabels=labels,
    yticklabels=labels,
    cbar_kws={'label': 'Mean Jaccard similarity'},
)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=6, rotation=45, ha='right')
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=6)
plt.savefig(f'{path_out}/avg_cooccurrence_clustermap.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Topic-specific co-occurrence variability

In [ ]:
# For each motif pair, compute variance of Jaccard across topics
pair_data = []
for i, m1 in enumerate(common_motifs):
    for j, m2 in enumerate(common_motifs):
        if j <= i:
            continue
        vals = []
        for topic in cooccur_matrices:
            mat = cooccur_matrices[topic]
            if m1 in mat.index and m2 in mat.index:
                vals.append(mat.loc[m1, m2])
            else:
                vals.append(0.0)
        pair_data.append({
            'motif1': m1, 'motif2': m2,
            'annotation1': annot_short.get(m1, m1),
            'annotation2': annot_short.get(m2, m2),
            'mean_jaccard': np.mean(vals),
            'std_jaccard': np.std(vals),
            'max_jaccard': np.max(vals),
            'cv': np.std(vals) / np.mean(vals) if np.mean(vals) > 0.01 else 0,
        })

pairs_df = pd.DataFrame(pair_data).sort_values('std_jaccard', ascending=False)
pairs_df.to_csv(f'{path_out}/motif_pair_cooccurrence.csv', index=False)

print('Top variable motif pairs (topic-specific co-occurrence):')
for _, row in pairs_df.head(15).iterrows():
    print(f"  {row['annotation1']:12s} + {row['annotation2']:12s}  "
          f"mean={row['mean_jaccard']:.3f}  std={row['std_jaccard']:.3f}")

print(f'\nTop consistently co-occurring pairs (high mean, low variance):')
consistent = pairs_df[(pairs_df['mean_jaccard'] > 0.05) & (pairs_df['cv'] < 0.3)]
for _, row in consistent.sort_values('mean_jaccard', ascending=False).head(10).iterrows():
    print(f"  {row['annotation1']:12s} + {row['annotation2']:12s}  "
          f"mean={row['mean_jaccard']:.3f}  cv={row['cv']:.2f}")

# Per-topic co-occurrence grid

In [ ]:
# Grid of co-occurrence heatmaps (select top topics by category diversity)
n_show = min(20, len(topic_list))
topics_to_show = topic_list[:n_show]

n_cols = 4
n_rows = (n_show + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4.5))
axes = axes.flatten()

for idx, topic in enumerate(topics_to_show):
    ax = axes[idx]
    mat = cooccur_matrices[topic].reindex(index=common_motifs, columns=common_motifs, fill_value=0)
    im = ax.imshow(mat.values, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.3)
    ax.set_xticks(range(len(common_motifs)))
    ax.set_xticklabels(labels, fontsize=4, rotation=90)
    ax.set_yticks(range(len(common_motifs)))
    ax.set_yticklabels(labels, fontsize=4)
    cat = topic_cat_dict.get(topic, 'unknown')
    cat_color = cat_colors.get(cat, '#999999')
    ax.set_title(f'{topic} ({cat})', fontsize=9, fontweight='bold', color=cat_color)

for idx in range(n_show, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig(f'{path_out}/cooccurrence_grid.pdf', dpi=200, bbox_inches='tight')
print(f'Saved grid: {n_show} topics, {len(common_motifs)} motifs')
plt.show()

# Save

In [ ]:
avg_cooccur.to_csv(f'{path_out}/avg_cooccurrence_matrix.csv')
print(f'Saved to {path_out}/')

# DONE!

---